In [1]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.current_device()}")  # GPU 사용 시 0 이상 출력

CUDA available: True
Device: 0


In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np

workModel = YOLO("/home/tm/deeplearning-repo-4/deep_learning/data/weights/seven_class_segmentation.pt")
helmetModel = YOLO("/home/tm/deeplearning-repo-4/deep_learning/data/weights/last_helmet_detection.pt")
image = cv2.imread("/home/tm/Downloads/Pasted image (2).png")
def predictEvent(self, img):
        imgrgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        poseResults = self.pose.process(imgrgb)
        detectedFires = self.fireDetection.predict(img, conf=0.7, verbose=False)
        workResults = self.workModel.predict(img, conf=0.5, verbose=False)
        
        newImg = img.copy()
        if poseResults.pose_landmarks:
            #self.mpDrawing.draw_landmarks(newImg, poseResults.pose_landmarks, self.mpPose.POSE_CONNECTIONS)
            
            landmarks = poseResults.pose_landmarks.landmark
            
            # Draw box on person
            h, w, _ = img.shape
            xCoords = [landmark.x * w for landmark in landmarks]
            yCoords = [landmark.y * h for landmark in landmarks]
            xMin, xMax = int(min(xCoords)), int(max(xCoords))
            yMin, yMax = int(min(yCoords)), int(max(yCoords))
            
            padding = 20
            xMin = max(0, xMin - padding)
            yMin = max(0, yMin - padding)
            xMax = min(w, xMax + padding)
            yMax = min(h, yMax + padding)
            
            shoulderXY = [(int(landmarks[i].x * img.shape[1]), int(landmarks[i].y * img.shape[0])) 
                          for i in [11, 12] if landmarks[i].visibility > 0.5]  # Left/Right Shoulder
            hipXY = [(int(landmarks[i].x * img.shape[1]), int(landmarks[i].y * img.shape[0])) 
                     for i in [23, 24] if landmarks[i].visibility > 0.5]  # Left/Right Hip
            
            if len(shoulderXY) > 0 and len(hipXY) > 0:             
                shoulderMid = np.mean(shoulderXY, axis=0).astype(int)
                hipMid = np.mean(hipXY, axis=0).astype(int)
                slope = abs((shoulderMid[1] - hipMid[1]) / (shoulderMid[0] - hipMid[0] + 1e-6))
                
                if slope < 0.3:
                    cv2.rectangle(newImg, (xMin, yMin), (xMax, yMax), (0, 255, 255), 2)
                    if self.detectedTime is None:
                        self.detectedTime = time.time()
                    else:
                        durationTime = time.time() - self.detectedTime
                        print(durationTime)
                        if durationTime >= 5:
                            self.sendDetectCommand(0x31, 1, "사고", "쓰러짐")
                        
        else:
            self.detectedTime = None
            self.sendDetectCommand(0x30, 1, "사고", "쓰러짐")
            
        if len(detectedFires[0].boxes) > 0:
            self.sendDetectCommand(0x31, 1, "사고", "화재")
            for box in detectedFires[0].boxes:
                
                xyxy = box.xyxy
                cv2.rectangle(newImg, (int(xyxy[0][0]), int(xyxy[0][1])), (int(xyxy[0][2]), int(xyxy[0][3])), (0, 0, 255), 2)
        else:
            self.sendDetectCommand(0x30, 1, "사고", "화재")
        
        if workResults[0].masks is not None:
            masks = workResults[0].masks.xy
        
            detectedClass = [workResults[0].names[int(cls)] for cls in workResults[0].boxes.cls]
            
            workerMasks = [masks[i] for i, cls in enumerate(detectedClass) if cls == "WO-01"]
        
            if "WO-03" in detectedClass:
                ladderIdx = detectedClass.index("WO-03")
                ladderPolygon = masks[ladderIdx]
                _, ladderY, _, ladderH = cv2.boundingRect(ladderPolygon)
                
                if "WO-01" in detectedClass:
                    helmetResults = self.helmetModel.predict(img, verbose=False, conf=0.5)
                    if len(helmetResults[0].boxes) > 0:
                        names = [helmetResults[0].names[cls.item()] for cls in helmetResults[0].boxes.cls.int()]
                        if "head" in names:
                            print("헬멧 미착용")
                            self.sendDetectCommand(0x31, 1, "사다리 작업 위반", "안전모")
                        else:
                            self.sendDetectCommand(0x30, 1, "사다리 작업 위반", "안전모")
                    isLadderViolation = False
                    for worker in workerMasks:
                        workerPolygon = np.array(worker, np.int32)
                        x, workerY, _, workerH = cv2.boundingRect(workerPolygon)
                        if workerY + workerH - ladderY < ladderH * 0.1:
                            print("NO")
                            cv2.putText(newImg, "NO", (x, workerY), cv2.FONT_HERSHEY_COMPLEX, 3, (255, 0, 0), 2)
                        else:
                            print("YES")
                            cv2.putText(newImg, "YES", (x, workerY), cv2.FONT_HERSHEY_COMPLEX, 3, (255, 0, 0), 2)
                            isLadderViolation = True
                    if isLadderViolation:
                        self.sendDetectCommand(0x31, 1, )
                            
            elif "SO-24" in detectedClass:
                rightClass = ["WO-23", "SO-40"]
                for cls in rightClass:
                    if cls not in detectedClass:
                        print(cls, " need!!!")
                    
            elif "SO-28" in detectedClass:
                rightClass = ["SO-20", "SO-40"]
                for cls in rightClass:
                    if cls not in detectedClass:
                        print(cls, " need!!!")
                helmetResults = self.helmetModel.predict(img, verbose=False, conf=0.5)
                if len(helmetResults[0].boxes) > 0:
                    names = [helmetResults[0].names[cls.item()] for cls in helmetResults[0].boxes.cls.int()]
                    if "head" in names:
                        print("헬멧 미착용")
            elif "WO-01" in detectedClass:
                helmetResults = self.helmetModel.predict(img, verbose=False, conf=0.5)
                if len(helmetResults[0].boxes) > 0:
                    names = [helmetResults[0].names[cls.item()] for cls in helmetResults[0].boxes.cls.int()]
                    if "head" in names:
                        print("헬멧 미착용")
            for idx, mask in enumerate(masks):
                # 다각형 좌표를 numpy 배열로 변환
                polygon = np.array(mask, np.int32)
                
                # 바운딩 박스 계산
                x, y, w, h = cv2.boundingRect(polygon)
                cv2.putText(newImg, detectedClass[idx], (x, y), cv2.FONT_HERSHEY_COMPLEX, 2, (255, 0, 0), 2)
                
                # 이미지에 바운딩 박스 그리기
                cv2.rectangle(newImg, (x, y), (x + w, y + h), (0, 255, 0), 2)  # 초록색 박스
        
        return newImg

predictEvent()
cv2.imshow("Frame", image)
cv2.waitKey(0)

cv2.destroyAllWindows()



0: 640x288 1 SO-24, 75.9ms
Speed: 1.0ms preprocess, 75.9ms inference, 1.8ms postprocess per image at shape (1, 3, 640, 288)
WO-23  need!!!
SO-40  need!!!


In [13]:
from ultralytics import YOLO
import numpy as np
import cv2
workModel = YOLO("/home/tm/deeplearning-repo-4/deep_learning/data/weights/seven_class_segmentation.pt").to("cuda")
results = workModel.predict("/home/tm/Downloads/N용접.jpg")

idx = [results[0].names[int(cls)] for cls in results[0].boxes.cls].index("SO-24")
polygon = np.array(results[0].masks.xy[idx], np.int32)
x, y, w, h = cv2.boundingRect(polygon)
x, y, w, h


image 1/1 /home/tm/Downloads/N용접.jpg: 384x640 1 SO-40, 1 WO-23, 1 SO-24, 9.1ms
Speed: 3.5ms preprocess, 9.1ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)


(450, 594, 103, 82)

In [1]:
import torch

print("CUDA 지원 여부:", torch.cuda.is_available())  
print("GPU 개수:", torch.cuda.device_count())  
if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))
    print("CUDA 버전:", torch.version.cuda)

CUDA 지원 여부: True
GPU 개수: 1
GPU 이름: NVIDIA GeForce RTX 3060 Laptop GPU
CUDA 버전: 12.4


In [5]:
from DbController import DbController 
dbCon = DbController("localhost", "root", "5315", "tfdb")
dbCon.connect()
dbCon.setCursor(True)

event = ["사고", "쓰러짐", "화재"]
_type = event[0]
tid = dbCon.getData(f"select tid from EventType where typeName = '{_type}'")[0][0]

eventList = event[1:]
if tid != 5:
    for each in eventList:
        aid = 3
        sid = dbCon.getData(f"select eid from Equipment where equipName = '{each}'")[0][0]
        values = (1, tid, sid, aid, filepath)
        sql = "insert into Report (RID, TID, SID, AID, imgPath) values (%s, %s, %s)"
        dbCon.myCursor.excute(sql, values)
else:
    for each in eventList:
        aid = dbCon.getData(f"select aid from Accident where accidentName = '{each}'")[0][0]
        print(aid)
        
        #values = (1, tid, aid, filepath)
        #sql = "insert into Report (RID, TID, AID, imgPath) values (%s, %s, %s)"
        #dbCon.myCursor.excute(sql, values)
    


1
2


In [13]:
from ultralytics import YOLO
import cv2
import numpy as np
import mediapipe as mp
import torch

workModel = YOLO("/home/tm/deeplearning-repo-4/deep_learning/data/weights/seven_class_segmentation.pt")
helmetModel = YOLO("/home/tm/deeplearning-repo-4/deep_learning/data/weights/last_helmet_detection.pt")
image = cv2.imread("/home/tm/Downloads/시나리오/2최상단1.jpg")
def predictEvent(img):
        mpPose = mp.solutions.pose
        imgrgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        pose = mpPose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)
        mpDrawing = mp.solutions.drawing_utils
        device = "cuda" if torch.cuda.is_available() else "cpu"
        poseResults = pose.process(imgrgb)
        fireDetection = YOLO("fire_detection.pt").to(device)
        detectedFires = fireDetection.predict(img, conf=0.7, verbose=False)
        workResults = workModel.predict(img, conf=0.5, verbose=False)
        
        newImg = img.copy()
        if poseResults.pose_landmarks:
            mpDrawing.draw_landmarks(newImg, poseResults.pose_landmarks, mpPose.POSE_CONNECTIONS)
            
            landmarks = poseResults.pose_landmarks.landmark
            
                        
        if len(detectedFires[0].boxes) > 0:

            for box in detectedFires[0].boxes:
                
                xyxy = box.xyxy
                cv2.rectangle(newImg, (int(xyxy[0][0]), int(xyxy[0][1])), (int(xyxy[0][2]), int(xyxy[0][3])), (0, 0, 255), 2)


        
        if workResults[0].masks is not None:
            masks = workResults[0].masks.xy
        
            
            for idx, mask in enumerate(masks):
                # 다각형 좌표를 numpy 배열로 변환
                polygon = np.array(mask, np.int32)
                
                # 바운딩 박스 계산
                x, y, w, h = cv2.boundingRect(polygon)
                
                # 이미지에 바운딩 박스 그리기
                cv2.rectangle(newImg, (x, y), (x + w, y + h), (0, 255, 0), 2)  # 초록색 박스
        
        return newImg

image = predictEvent(image)
image = cv2.resize(image, (1680, 960))
cv2.imshow("Frame", image)
cv2.waitKey(0)

cv2.destroyAllWindows()


I0000 00:00:1743750862.501218   71511 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1743750862.503389   72565 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 24.2.8-1ubuntu1~24.04.1), renderer: Mesa Intel(R) UHD Graphics (CML GT2)
W0000 00:00:1743750862.566590   72553 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1743750862.616433   72558 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
